In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
import os
import seaborn as sns
import pandas as pd
from collections import Counter
from sklearn.cluster import OPTICS
from sklearn.metrics import pairwise_distances
from sklearn.decomposition import PCA

## Caricamento dei files

In [ ]:
# Imposta il percorso della cartella contenente i file pkl
path = '/content/drive/MyDrive/cold_heads_monitoring/Dataset'
arrays = []
data_list = []
lens = []

# Itera su tutti i file pkl nella cartella e caricali in una lista
for filename in os.listdir(path):
    if filename.endswith('.pkl'):
        with open(os.path.join(path, filename), 'rb') as f:
            # Carica l'array NumPy dal file e seleziona solo il secondo canale
            arr = np.asarray(pickle.load(f))[:, 1:2, :]
            lens.append(len(arr))
            arrays.append(arr)
            data_list.append(os.path.splitext(filename)[0])

print(data_list)
print(lens)

In [ ]:
all_data = arrays[2]
#arrays = [np.array(arrays[6]), np.array(arrays[4]), np.array(arrays[2]), np.array(arrays[1]), np.array(arrays[5]), np.array(arrays[0])]

## Bilanciamento del dataset

In [ ]:
# Calcola la dimensione minima tra gli array nella lista
#min_len = min([arr.shape[0] for arr in arrays])
min_len = 2000

# Bilancia gli array nella lista
balanced_arrays = [arr[:min_len] for arr in arrays]

# Unisci gli array bilanciati in un unico array NumPy
data = np.concatenate(balanced_arrays, axis=0)

data = np.squeeze(data,1)
data.shape

In [ ]:
def calculate_features(window):
    # Calcola la media del segnale nella finestra
    mean = np.mean(window)

    # Calcola la deviazione standard del segnale nella finestra
    std_dev = np.std(window)

    # Calcola il massimo valore nella finestra
    max_val = np.max(window)

    # Calcola il minimo valore nella finestra
    min_val = np.min(window)

    # Calcola la differenza tra il massimo e il minimo
    peak_to_peak = max_val - min_val

    return [mean, std_dev, max_val, min_val, peak_to_peak]

In [ ]:
def normalize_data(data):
    # Calcola la media e la deviazione standard lungo ciascuna feature (colonna)
    mean = np.mean(data, axis=0)
    std_dev = np.std(data, axis=0)

    # Applica la standardizzazione (z-score normalization)
    normalized_data = (data - mean) / std_dev

    return normalized_data

In [ ]:
pca = PCA(n_components=2)

def preprocessing(data):
    # Lunghezza della finestra e sovrapposizione
    window_size = 30
    overlap = 2

    # Calcola tutte le finestre scorrevoli
    windows = []
    for signal in data:
        n = len(signal)
        indices = np.arange(0, n - window_size + 1, overlap)
        windows.extend(signal[i:i+window_size] for i in indices)

    # Calcola le features per tutte le finestre
    features = np.array([calculate_features(window) for window in windows])

    # Normalizza i dati
    #normalized_data = normalize_data(features)

    # Riduzione della dimensionalità
    #dataset = pca.fit_transform(dataset)

    # Mischia casualmente i dati
    np.random.shuffle(features)

    return features


## Training DBSCAN

In [ ]:
dataset = preprocessing(data)[:1000]
# crea il modello di clustering con DBSCAN
optics = OPTICS(min_samples=10, xi=0.79, min_cluster_size=0.05) #default 0.8
optics.fit(dataset)
clustering_labels = optics.labels_

In [ ]:
# Visualizza i risultati del clustering
n_clusters = len(set(clustering_labels)) - (1 if -1 in clustering_labels else 0)
print("Numero di cluster trovati:", n_clusters)

# Visualizza i punti anomali (cluster con etichetta -1)
n_anomalies = list(clustering_labels).count(-1)
print("Numero di punti anomali trovati:", n_anomalies)

In [ ]:
# Creare un DataFrame per i dati e le etichette di clustering
df = pd.DataFrame({'X': dataset[:, 0], 'Y': dataset[:, 1], 'Cluster': clustering_labels})

# Plottiamo i cluster usando Seaborn
sns.scatterplot(data=df, x='X', y='Y', hue='Cluster', palette='viridis', alpha=0.5)
plt.show()

#Inference

In [ ]:
# nuovo_campione = il tuo nuovo campione
# data = i tuoi dati di addestramento
# clustering_labels = le etichette di clustering assegnate ai dati di addestramento

def distance(nuovo_campione, data, clustering_labels, metric='chebyshev'):
    # Calcola il centroide di ciascun cluster
    cluster_centroids = []
    for label in np.unique(clustering_labels):
        cluster_points = data[clustering_labels == label]
        centroid = np.mean(cluster_points, axis=0)
        cluster_centroids.append(centroid)

    cluster_centroids = np.array(cluster_centroids)

    # Calcola la distanza tra il nuovo campione e i centroidi dei cluster
    distanze = pairwise_distances(nuovo_campione.reshape(1, -1), cluster_centroids, metric=metric)

    # Identifica il cluster più vicino
    cluster_più_vicino = np.argmin(distanze)
    return cluster_più_vicino

In [ ]:
# Calcola la dimensione minima tra gli array nella lista
#min_len = min([arr.shape[0] for arr in arrays])
min_len = 2000

inference_data = all_data[:min_len]

inference_data = np.squeeze(inference_data,1)
inference_data.shape

In [ ]:
inference_data = preprocessing(inference_data)

In [ ]:
# Creare un DataFrame per i dati e le etichette di clustering
df2 = pd.DataFrame({'X': inference_data[:, 0], 'Y': inference_data[:, 1]})

# Plottiamo i cluster usando Seaborn
sns.scatterplot(data=df2, x='X', y='Y', palette='viridis', alpha=0.5)
sns.scatterplot(data=df, x='X', y='Y', hue='Cluster', palette='viridis', alpha=0.5)
plt.show()

In [ ]:
result = []
for elem in inference_data:
  result.append(distance(elem, dataset, clustering_labels))
count = Counter(result)
elemento_piu_ripetuto = count.most_common(1)[0][0]
print(f"Cluster più vicino al nuovo campione: {elemento_piu_ripetuto}")

In [ ]:
# Salva il modello con pickle
with open('dbscan_model.pkl', 'wb') as model_file:
    pickle.dump(optics, model_file)

In [ ]:
new_labels = optics.fit_predict(data)
print(max(new_labels))